In [ ]:

# 11. Suitability & Robustness Check

from sklearn.inspection import permutation_importance
from sklearn.model_selection import learning_curve

print("\n" + "="*30)
print("ANALYZING DATASET SUITABILITY")
print("="*30)

# 1. Permutation Importance (Is the model logic sound?)
# This checks if the features actually contribute or if the model is lucky.
perm_importance = permutation_importance(best_xgb, X_test, y_test, n_repeats=10, random_state=42)
sorted_idx = perm_importance.importances_mean.argsort()[-10:]

plt.figure(figsize=(10, 6))
plt.barh(poly_feature_names[sorted_idx], perm_importance.importances_mean[sorted_idx])
plt.xlabel("Permutation Importance (Drop in Accuracy)")
plt.title("Top 10 Reliable Features (Suitability Check)")
plt.show()

# 2. Learning Curve (Is the dataset size sufficient?)
train_sizes, train_scores, test_scores = learning_curve(
    best_xgb, X_resampled, y_resampled, cv=cv, scoring='accuracy',
    n_jobs=-1, train_sizes=np.linspace(0.1, 1.0, 5)
)

train_mean = np.mean(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, 'o-', label="Training Score")
plt.plot(train_sizes, test_mean, 's-', label="Cross-Validation Score")
plt.xlabel("Training Examples")
plt.ylabel("Accuracy Score")
plt.title("Learning Curve: Is the Dataset a Good Suit?")
plt.legend()
plt.grid(True)
plt.show()

# 3. Final Verdict Printout
gap = train_mean[-1] - test_mean[-1]
print(f"\nModel Suitability Verdict:")
if gap > 0.15:
    print(" HIGH OVERFIT: The dataset might be too small or noisy for a model this complex.")
elif test_mean[-1] < 0.65:
    print(" UNDERFIT: The features in this dataset don't correlate strongly with potability.")
else:
    print("GOOD SUIT: The model generalizes well and the feature relationships are stable.")